# LangGraph Command Pattern
## Jump to Any Node — Command-Driven Graph Navigation
⏱ ~10 min

LangGraph's **`Command`** primitive lets a node return routing decisions and state updates in a single object — no `add_conditional_edges` required. Instead of a separate routing function wired through a mapping dict, the node itself decides where to go next and what to write to state, all in one `return Command(goto=..., update={...})`.

By the end of this session you will understand *why* the Command pattern exists, *how* it differs from conditional edges, and *how* to apply it for dynamic dispatch, multi-hop routing, and fan-out with `Send`.

---

### Workshop Roadmap

| # | Topic |
|---|-------|
| 1 | **Concepts** — What is Command and why does it exist? |
| 2 | **State Design** — TypedDict, routing slots, and reducers |
| 3 | **First Command Graph** — Router + four specialist handlers |
| 4 | **Dynamic goto** — `str`, `Send`, and list targets |
| 5 | **Fallback Routing** — Graceful handling of unknown categories |
| 6 | **Multi-hop Routing** — Chaining Commands across nodes |
| 7 | **Debugging** — Streaming events and graph validation |
| ★ | **Advanced** — Confidence routing + fan-out (bonus) |

---

### Prerequisites
- Python 3.10+ with `langgraph`, `langchain-openai`, `python-dotenv` installed
- An `OPENAI_API_KEY` in `.env` (or Colab Secrets)

### Key References
> LangGraph `Command` API: https://langchain-ai.github.io/langgraph/reference/types/#langgraph.types.Command  
> LangGraph `Send` API (fan-out): https://langchain-ai.github.io/langgraph/reference/types/#langgraph.types.Send  
> LangGraph conceptual guide — Command: https://langchain-ai.github.io/langgraph/concepts/low_level/#command  
> LangGraph how-to — dynamic routing: https://langchain-ai.github.io/langgraph/how-tos/command/

In [ ]:
# Install dependencies — runs only on Google Colab.
# Local users: your .venv already has everything from requirements.txt.
import sys


def _in_colab():
    try:
        import google.colab

        return True
    except ImportError:
        return False


if _in_colab():
    import subprocess

    subprocess.run(
        [
            sys.executable,
            "-m",
            "pip",
            "install",
            "-q",
            "langgraph",
            "langchain-openai",
            "langchain-core",
            "python-dotenv",
        ]
    )
    print("Colab install complete.")
else:
    print("Local environment detected — skipping install.")

In [ ]:
import os

# ── API key ───────────────────────────────────────────────────────────────────
# Colab: set in Secrets panel (left sidebar key icon)
# Local: create a .env file containing  OPENAI_API_KEY=sk-...
try:
    from google.colab import userdata

    os.environ["OPENAI_API_KEY"] = userdata.get("OPENAI_API_KEY")
except ImportError:
    from dotenv import load_dotenv

    load_dotenv()

# ── Sanity check ──────────────────────────────────────────────────────────────
key = os.environ.get("OPENAI_API_KEY", "")
assert key.startswith("sk-"), (
    "OPENAI_API_KEY missing or malformed. "
    "Local: add it to .env. Colab: add it in the Secrets panel."
)
print("OPENAI_API_KEY present: True (starts with sk-)")

---

## Part 1 — What Is Command and Why Does It Exist?

---

### The Problem With `add_conditional_edges`

The standard LangGraph routing pattern separates the routing *decision* from the node that *makes* it:

```python
# Node makes a decision and writes it to state
def router(state):
    route = classify(state["task"])
    return {"route": route}           # state update only

# A separate function reads state and returns the next node name
def pick_route(state) -> str:
    return state["route"]

# A mapping dict translates the string to a registered node
graph.add_conditional_edges(
    "router",
    pick_route,
    {"code": "code", "explain": "explain", "math": "math"},
)
```

This works, but routing logic is spread across three places: the node, the edge function, and the mapping dict. Adding a new category means updating all three.

---

### The Command Solution

**`Command`** collapses those three into one return value:

```python
from langgraph.types import Command

def router(state) -> Command:
    route = classify(state["task"])   # any logic here
    return Command(
        goto=route,                   # where to go next
        update={"route": route},      # what to write to state
    )
# No add_conditional_edges needed — ever.
```

Adding a new category now means: (1) add the node, (2) include the new name as a possible `goto` value. Nothing else.

---

### Command vs `add_conditional_edges` — Diagram

```
CONDITIONAL EDGES STYLE
──────────────────────────────────────────────────────────────

  ┌──────────┐   returns {"route": "code"}   ┌───────────────┐
  │  router  │──────────────────────────────►│ state["route"]│
  └──────────┘                               └───────┬───────┘
                                                     │ pick_route(state)
                       ┌─────────────────────────────┘
                       │  mapping: {"code": "code", "explain": "explain", ...}
                       ▼
                 ┌──────────┐
                 │   code   │
                 └──────────┘


COMMAND STYLE
──────────────────────────────────────────────────────────────

  ┌──────────┐   Command(goto="code",          ┌──────────┐
  │  router  │        update={"route": "code"})│   code   │
  └──────────┘──────────────────────────────►  └──────────┘
               (routing + state update in one object,
                computed at runtime — no mapping dict)
```

---

### Key Vocabulary

| Term | Definition |
|------|------------|
| **`Command`** | Return value from a node that controls both routing (`goto`) and state (`update`) |
| **`goto`** | Target node — a `str`, a `Send` object, or a `list` of either |
| **`update`** | Dict of keys to write into graph state before the next node runs |
| **`Send`** | A `goto` variant that also packages per-branch input for fan-out |
| **`add_conditional_edges`** | The older pattern — separate routing function + mapping dict |
| **Dynamic dispatch** | Routing to a node whose name is computed at runtime, not fixed at build time |
| **Fan-out** | Dispatching the same task to multiple nodes in parallel via a list of `Send` objects |

In [ ]:
# ─── 1-A: Import Command and inspect its interface ────────────────────────────
import inspect

from langgraph.types import Command, Send

print("Command signature:")
print(inspect.signature(Command))
print()

# Create a simple Command to see its repr
example_cmd = Command(goto="code", update={"route": "code"})
print(f"Example Command : {example_cmd}")
print(f"  .goto         : {example_cmd.goto}")
print(f"  .update       : {example_cmd.update}")
print()

# goto can be a list for multi-target routing
fan_out_cmd = Command(goto=["node_a", "node_b"], update={"status": "dispatched"})
print(f"Fan-out Command : {fan_out_cmd}")

In [ ]:
# ─── 1-B: The three forms of goto ────────────────────────────────────────────
#
# goto accepts three forms:
#   1. str               — go to a named node
#   2. Send(node, input) — go to a node AND pass extra input (fan-out)
#   3. list[str | Send]  — multiple targets in parallel

from langgraph.types import Command, Send

# Form 1 — simple string
cmd_str = Command(goto="explain", update={"route": "explain"})

# Form 2 — Send packages a node name + a custom input dict
cmd_send = Command(
    goto=Send("code", {"task": "Write a hello-world function", "language": "python"})
)

# Form 3 — list of Send objects (parallel fan-out)
cmd_list = Command(
    goto=[
        Send("translate", {"phrase": "Good morning", "language": "French"}),
        Send("translate", {"phrase": "Good morning", "language": "Spanish"}),
    ],
    update={"status": "fan-out dispatched"},
)

print("Form 1 — str goto:")
print(f"  {cmd_str}")
print()
print("Form 2 — Send goto:")
print(f"  {cmd_send}")
print()
print("Form 3 — list goto (fan-out):")
print(f"  {cmd_list}")

---

## Part 2 — State Design for Command Graphs

---

### What Belongs in State?

LangGraph state is a **shared blackboard** — every node reads from it and can write to it (via `Command.update` or by returning a plain dict). Design state with exactly the fields each node needs.

For a Command-pattern router a minimal state has:
- **Input field** — the data to classify (`task: str`)
- **Routing slot** — the decision the router made (`route: str`) — useful for inspection
- **Output field** — what the handler produces (`result: str`)

```
State flow through the graph:

  invoke({ task: "Write fibonacci", route: "", result: "" })
         │
         ▼
  router node
    Command(goto="code", update={"route": "code"})
    state after: { task: "Write fibonacci", route: "code", result: "" }
         │
         ▼
  code_handler node
    return {"result": "def fib(n): ..."}
    state after: { task: "Write fibonacci", route: "code", result: "def fib..." }
         │
         ▼
  END
```

---

### Public vs Private State

| | Public state (`TypedDict`) | Private state (per-node, via `Send`) |
|---|---|---|
| **Visible to** | All nodes in graph | Only the receiving node |
| **Use case** | Shared routing slots, results | Fan-out branches with custom inputs |
| **Written by** | Any node via `Command.update` | `Send(node, {"key": value})` |

### Reducers — Collecting Parallel Results

When multiple branches write to the same state field, you need a **reducer** so their results are merged rather than the last write winning:

```python
import operator
from typing import Annotated

class FanOutState(TypedDict):
    summaries: Annotated[list, operator.add]  # each branch appends; results accumulate
```

In [ ]:
# ─── 2-A: Define state and trace a simulated execution ────────────────────────
from typing import TypedDict


# TypedDict gives LangGraph typed access to state keys; fields become the
# shared blackboard every node reads from and Command.update writes to.
class CommandState(TypedDict):
    task: str    # input: the user's request
    route: str   # routing slot: populated by the router node via Command.update
    result: str  # output: populated by the handler node


# Simulate what Command.update does to state (LangGraph applies this automatically)
initial_state: CommandState = {"task": "Write fibonacci in Python", "route": "", "result": ""}
print("Initial state:")
for k, v in initial_state.items():
    print(f"  {k!r}: {v!r}")

# Router returns Command — LangGraph merges .update into state
from langgraph.types import Command
router_cmd = Command(goto="code", update={"route": "code"})
after_router = {**initial_state, **router_cmd.update}
print(f"\nAfter router (Command.update applied):")
for k, v in after_router.items():
    print(f"  {k!r}: {v!r}")

# Handler returns plain dict — LangGraph merges it too
handler_output = {"result": "def fib(n): return n if n <= 1 else fib(n-1) + fib(n-2)"}
final_state = {**after_router, **handler_output}
print(f"\nAfter handler (return dict applied):")
for k, v in final_state.items():
    print(f"  {k!r}: {v!r}")

In [ ]:
# ─── 2-B: Reducer demo — collecting results from parallel branches ────────────
import operator
from typing import Annotated


# Annotated[list, operator.add] is the reducer — without it, parallel branches
# overwrite each other; only the last writer's result survives.
class FanOutState(TypedDict):
    items: list
    results: Annotated[list, operator.add]  # reducer: merges all branch outputs


# Without reducer: last write wins (wrong for parallel branches)
no_reducer = {"results": ["branch_a"]}
no_reducer.update({"results": ["branch_b"]})  # branch_a is lost!
print(f"Without reducer: {no_reducer['results']}  ← branch_a lost")

# With reducer: all branch outputs are concatenated
# LangGraph applies operator.add when multiple nodes write to an Annotated field
simulated_merge = operator.add(["branch_a"], ["branch_b"])
print(f"With reducer:    {simulated_merge}  ← both preserved")

---

## Part 3 — Building the First Command Graph

---

### Architecture

```
START
  │
  ▼
┌──────────┐   Command(goto="code"|"explain"|"math"|"creative")
│  router  │────────────────────────────────────────────────────►
└──────────┘   (LLM classifies task — no mapping dict, no edge function)
  │
  ├──► ┌──────────┐
  │    │   code   │ — expert programmer system prompt
  │    └────┬─────┘
  │         ▼
  ├──► ┌──────────┐
  │    │ explain  │ — teacher system prompt
  │    └────┬─────┘
  │         ▼
  ├──► ┌──────────┐
  │    │   math   │ — math tutor system prompt
  │    └────┬─────┘
  │         ▼
  └──► ┌───────────┐
       │ creative  │ — creative writer system prompt
       └────┬──────┘
            ▼
           END
```

**`add_conditional_edges` is never called.** Routing happens entirely inside `router()` via the `Command` return value. Each handler node connects to `END` with a plain `add_edge`.

In [ ]:
# ─── 3-A: Define LLM, prompts, and valid routes ───────────────────────────────
from langchain_core.messages import HumanMessage, SystemMessage
from langchain_openai import ChatOpenAI

# temperature=0.3: low enough for consistent classification, nonzero so
# borderline tasks aren't locked to one token arbitrarily.
llm = ChatOpenAI(model="gpt-4o-mini", temperature=0.3)

ROUTER_PROMPT = """Classify this task into exactly one category: code, explain, math, or creative.

Task: {task}

Respond with only the category name (no punctuation, no explanation)."""

PROMPTS = {
    "code":     "You are an expert programmer. Write clean, working code for: {task}",
    "explain":  "You are a great teacher. Explain clearly in 3-5 sentences: {task}",
    "math":     "You are a math tutor. Solve step by step: {task}",
    "creative": "You are a creative writer. Respond with creativity and humor: {task}",
}

VALID_ROUTES = set(PROMPTS.keys())

print(f"LLM           : {llm.model_name}")
print(f"Valid routes  : {sorted(VALID_ROUTES)}")
print(f"Router prompt : {ROUTER_PROMPT[:80]}...")

In [ ]:
# ─── 3-B: Define router node and handler factory ──────────────────────────────
from typing import TypedDict

from langgraph.types import Command


class CommandState(TypedDict):
    task: str
    route: str
    result: str


# Returning Command instead of a plain dict is the key distinction:
# Command bundles the routing decision WITH the state update in one object.
def router(state: CommandState) -> Command:
    """Classify the task and return a Command that routes to the right handler."""
    prompt = ROUTER_PROMPT.format(task=state["task"])
    result = llm.invoke([HumanMessage(content=prompt)])
    route = result.content.strip().lower()

    # Fallback: if the LLM returns something unexpected, default to explain
    if route not in VALID_ROUTES:
        print(f"  [router] unrecognised {route!r} — falling back to 'explain'")
        route = "explain"

    print(f"  [router] classified as: {route!r}")
    # Command: goto  = where to go next
    #          update = what to merge into state before the next node runs
    return Command(goto=route, update={"route": route})


def make_handler(category: str):
    """Factory: create a handler node function for each specialist category."""
    def handler(state: CommandState) -> dict:
        prompt = PROMPTS[category].format(task=state["task"])
        result = llm.invoke([SystemMessage(content=prompt)])
        return {"result": result.content}

    handler.__name__ = f"{category}_handler"
    return handler


print("Node functions defined:")
print("  router()        — classifies task, returns Command(goto=route)")
for cat in sorted(VALID_ROUTES):
    print(f"  {cat}_handler() — {PROMPTS[cat][:55]}...")

In [ ]:
# ─── 3-C: Build and compile the graph ────────────────────────────────────────
from langgraph.graph import END, StateGraph

builder = StateGraph(CommandState)

# Add the router node
builder.add_node("router", router)

# Add one handler node per category, each wired straight to END
for category in VALID_ROUTES:
    builder.add_node(category, make_handler(category))
    builder.add_edge(category, END)

# Entry point is the router
builder.set_entry_point("router")

# ────────────────────────────────────────────────────────────────────────────
# NOTE: add_conditional_edges is NOT called anywhere in this notebook.
# Command(goto=route) inside router() IS the conditional edge.
# ────────────────────────────────────────────────────────────────────────────

# compile() locks the graph topology — nodes and edges cannot change after
# this point. It also wires up any checkpointer for persistence.
app = builder.compile()

print("Graph compiled.")
print(f"Nodes: {sorted(builder.nodes.keys())}")
print("No add_conditional_edges called — Command handles routing internally.")

In [ ]:
# ─── 3-D: Run the graph on four different task types ─────────────────────────
TASKS = [
    "Write a Python function to calculate fibonacci numbers.",
    "Explain what a REST API is in simple terms.",
    "What is 42 * 17? Show your work.",
    "Tell me a short joke about programming.",
]

for task in TASKS:
    print(f"\nTASK: {task}")
    print("─" * 60)
    result = app.invoke({"task": task, "route": "", "result": ""})
    print(f"Route : {result['route']}")
    print(f"Result: {result['result'][:250]}")

---

## Part 4 — Dynamic `goto`: `str`, `Send`, and Lists

---

### Three Forms of `goto`

| Form | Syntax | Effect |
|------|--------|--------|
| **String** | `Command(goto="node_name")` | Route to a single named node |
| **Send** | `Command(goto=Send("node", {"k": "v"}))` | Route to a node AND pass custom per-branch input |
| **List** | `Command(goto=[Send(...), Send(...)])` | Fan-out to multiple nodes in parallel |

### When to Use Each

- **String** — most common. Classify a task and route to one specialist.
- **Send** — use when you need to pass *extra* data to the target that isn't already in state. Example: fan-out where each branch needs a different document slice.
- **List** — use when multiple nodes should run concurrently. Each node in the list gets the same state snapshot (plus its `Send`-specific input if using `Send`).

### How `Send` Works

```
documents = ["doc1", "doc2", "doc3"]

Command(
    goto=[
        Send("summarise", {"doc": "doc1"}),
        Send("summarise", {"doc": "doc2"}),
        Send("summarise", {"doc": "doc3"}),
    ]
)
```

Each `Send` creates an independent invocation of `summarise` with its own private input. Results are collected via an `Annotated[list, operator.add]` reducer on a shared state field.

In [ ]:
# ─── 4-A: String goto — self-contained minimal example ───────────────────────
from typing import TypedDict

from langgraph.graph import END, StateGraph
from langgraph.types import Command


class SimpleState(TypedDict):
    value: str
    branch: str


def brancher(state: SimpleState) -> Command:
    branch = "left" if len(state["value"]) > 5 else "right"
    return Command(goto=branch, update={"branch": branch})


def left_node(state: SimpleState) -> dict:
    return {"value": f"LEFT: {state['value']}"}


def right_node(state: SimpleState) -> dict:
    return {"value": f"RIGHT: {state['value']}"}


g = StateGraph(SimpleState)
g.add_node("brancher", brancher)
g.add_node("left", left_node)
g.add_node("right", right_node)
g.add_edge("left", END)
g.add_edge("right", END)
g.set_entry_point("brancher")
simple_app = g.compile()

for val in ["hi", "hello world"]:
    result = simple_app.invoke({"value": val, "branch": ""})
    print(f"Input: {val!r:15} → branch={result['branch']!r:7} → {result['value']}")

In [ ]:
# ─── 4-B: Send — fan-out where each branch gets custom input ─────────────────
# Use case: translate a phrase into multiple languages in parallel.
import operator
from typing import Annotated, TypedDict

from langgraph.types import Command, Send


class TranslationState(TypedDict):
    phrase: str
    language: str
    translations: Annotated[list, operator.add]  # reducer: accumulate all branch outputs


# Send(node, input) lets you pass custom state to each branch independently —
# unlike a plain str goto, which shares the same state with the target node.
def dispatcher(state: TranslationState) -> Command:
    """Fan out — one translator invocation per target language."""
    languages = ["French", "Spanish", "Japanese"]
    return Command(
        goto=[
            Send(
                "translate",
                {"phrase": state["phrase"], "language": lang, "translations": []}
            )
            for lang in languages
        ]
    )


def translate(state: TranslationState) -> dict:
    """Translate the phrase into the language specified in Send input."""
    result = llm.invoke([
        HumanMessage(
            content=f"Translate '{state['phrase']}' to {state['language']}. "
                    f"Reply with only the translation."
        )
    ]).content.strip()
    return {"translations": [f"{state['language']}: {result}"]}


t_builder = StateGraph(TranslationState)
t_builder.add_node("dispatcher", dispatcher)
t_builder.add_node("translate", translate)
t_builder.add_edge("translate", END)
t_builder.set_entry_point("dispatcher")
t_app = t_builder.compile()

result = t_app.invoke({"phrase": "Good morning", "language": "", "translations": []})
print("Fan-out translation results:")
for t in result["translations"]:
    print(f"  {t}")

---

## Part 5 — Fallback Routing and Defensive Patterns

---

### Why Fallback Routing Matters

A classifier that calls an LLM may return an unexpected value — the model might return `"Code"` (capitalised), `"coding"` (synonym), or a hallucinated category. Without a fallback, LangGraph raises a `ValueError` because the `goto` target doesn't exist as a registered node.

**Three strategies:**

| Strategy | How | When to use |
|----------|-----|-------------|
| **Normalise** | `route.lower().strip().rstrip(".,")` | Handles case/whitespace/punctuation mismatches |
| **Default fallback** | `if route not in VALID: route = "explain"` | Simple, always succeeds |
| **Fuzzy match** | Substring check: `if valid in route or route in valid` | When synonyms are common |

The first two cover almost all production cases. The third adds resilience when the classifier prompt is underspecified.

In [ ]:
# ─── 5-A: safe_route() — normalise + fuzzy match + fallback ──────────────────

VALID_ROUTES = {"code", "explain", "math", "creative"}


def safe_route(raw: str, fallback: str = "explain") -> str:
    """Normalise a raw LLM response into a valid route name."""
    normalised = raw.strip().lower().rstrip(".,!?")
    if normalised in VALID_ROUTES:
        return normalised
    # Try substring match (e.g. "coding" → "code")
    for valid in VALID_ROUTES:
        if valid in normalised or normalised in valid:
            print(f"  [safe_route] fuzzy match: {raw!r} → {valid!r}")
            return valid
    print(f"  [safe_route] unknown {raw!r} → fallback {fallback!r}")
    return fallback


test_cases = [
    "code",         # exact
    "Code",         # capitalised
    "MATH.",        # uppercase + punctuation
    "coding",       # synonym
    "science",      # unrelated
    "\n explain ",  # whitespace
]

print("safe_route() normalisation tests:")
for raw in test_cases:
    route = safe_route(raw)
    print(f"  {raw!r:22} → {route!r}")

In [ ]:
# ─── 5-B: Router with defensive normalisation wired in ────────────────────────
from langgraph.graph import END, StateGraph
from langgraph.types import Command


def robust_router(state: CommandState) -> Command:
    """Router that never raises on unexpected LLM output."""
    prompt = ROUTER_PROMPT.format(task=state["task"])
    raw = llm.invoke([HumanMessage(content=prompt)]).content
    route = safe_route(raw, fallback="explain")
    print(f"  [router] raw={raw!r}  →  route={route!r}")
    return Command(goto=route, update={"route": route})


robust_builder = StateGraph(CommandState)
robust_builder.add_node("router", robust_router)
for cat in VALID_ROUTES:
    robust_builder.add_node(cat, make_handler(cat))
    robust_builder.add_edge(cat, END)
robust_builder.set_entry_point("router")
robust_app = robust_builder.compile()

# Test with ambiguous input — should still route cleanly
test_task = "What is the boiling point of water?"
r = robust_app.invoke({"task": test_task, "route": "", "result": ""})
print(f"\nTask  : {test_task}")
print(f"Route : {r['route']}")
print(f"Result: {r['result'][:200]}")

---

## Part 6 — Multi-hop Routing

---

### Chaining Commands Across Multiple Nodes

A handler node can also return a `Command` instead of a plain dict — it routes to a *second* processing step rather than going straight to `END`. This creates a **multi-hop** execution path.

```
START → router ──Command──► code_handler ──Command──► reviewer ──► END
               └──Command──► explain_handler ────────────────────► END
```

The pattern is the same: any node can return `Command(goto=..., update={...})`. LangGraph follows the chain until it reaches `END`.

### Common Multi-hop Patterns

| Scenario | Multi-hop chain |
|----------|-----------------|
| Code quality pipeline | `router → code_handler → reviewer → END` |
| Document pipeline | `router → extractor → summariser → embedder → END` |
| Retry loop | `handler → validator` → if invalid → back to `handler` (with counter) |
| Multi-agent handoff | `supervisor → agent_a → agent_b → supervisor → END` |

In [ ]:
# ─── 6-A: Two-hop graph — router → code_handler → reviewer ───────────────────
from typing import TypedDict

from langgraph.graph import END, StateGraph
from langgraph.types import Command


class ReviewState(TypedDict):
    task: str
    route: str
    draft: str    # code_handler writes here
    result: str   # reviewer writes here


REVIEW_PROMPT = (
    "Review the following code for clarity and correctness. "
    "Add a one-sentence summary at the top.\n\nCode:\n{draft}"
)


def two_hop_router(state: ReviewState) -> Command:
    raw = llm.invoke([HumanMessage(content=ROUTER_PROMPT.format(task=state["task"]))]).content
    route = safe_route(raw, "explain")
    print(f"  [router] → {route!r}")
    return Command(goto=route, update={"route": route})


# Returning Command from a handler (not just the router) is what enables
# multi-hop chains — any node can redirect execution to the next step.
def code_handler_v2(state: ReviewState) -> Command:
    """Generate code draft, then chain to reviewer via Command."""
    draft = llm.invoke(
        [SystemMessage(content=PROMPTS["code"].format(task=state["task"]))]
    ).content
    print(f"  [code_handler] draft {len(draft)} chars → chaining to reviewer")
    # Return Command instead of plain dict to continue the chain
    return Command(goto="reviewer", update={"draft": draft})


def explain_handler_v2(state: ReviewState) -> dict:
    """Explain path goes straight to END — no review step."""
    result = llm.invoke(
        [SystemMessage(content=PROMPTS["explain"].format(task=state["task"]))]
    ).content
    return {"result": result, "draft": result}


def reviewer(state: ReviewState) -> dict:
    """Final review pass — reads draft, writes final result."""
    reviewed = llm.invoke(
        [SystemMessage(content=REVIEW_PROMPT.format(draft=state["draft"]))]
    ).content
    print(f"  [reviewer] complete ({len(reviewed)} chars)")
    return {"result": reviewed}


r_builder = StateGraph(ReviewState)
r_builder.add_node("router", two_hop_router)
r_builder.add_node("code", code_handler_v2)
r_builder.add_node("explain", explain_handler_v2)
r_builder.add_node("reviewer", reviewer)
r_builder.add_edge("reviewer", END)
r_builder.add_edge("explain", END)
r_builder.set_entry_point("router")
r_app = r_builder.compile()

task = "Write a Python function that returns the first N prime numbers."
print(f"Task: {task}\n{'-' * 60}")
out = r_app.invoke({"task": task, "route": "", "draft": "", "result": ""})
print(f"\nFinal result (first 400 chars):\n{out['result'][:400]}")

---

## Part 7 — Debugging Command Graphs

---

### Common Failure Modes

| Failure | Symptom | Fix |
|---------|---------|-----|
| **Invalid goto target** | `ValueError: Node 'xyz' not found` | Add fallback normalisation; check `VALID_ROUTES` against registered nodes |
| **Missing state key** | `KeyError: 'route'` | Ensure all state fields are present in the initial `invoke()` dict |
| **Reducer missing** | List field reset by last branch | Annotate with `Annotated[list, operator.add]` |
| **Infinite loop** | Graph never reaches END | Ensure every path eventually routes to END; use a retry counter |
| **Command ignored** | Routing doesn't happen | Node must return `Command(...)` not `{"route": ...}` |

### Debugging Checklist

1. Print `Command.goto` and `Command.update` inside the router before returning
2. Stream events with `app.stream(...)` to see each node's output in real time
3. Check `builder.nodes.keys()` to confirm every `goto` target is registered
4. Start with a manual simulation (like Part 2-A) before running the full graph

In [ ]:
# ─── 7-A: Stream graph events to trace execution ─────────────────────────────
# app.stream() yields one event dict per node — the best debugging tool.

task = "What is 12 factorial? Show your work."
print(f"Task: {task}\n{'-' * 60}")
print("Streaming execution events:\n")

# stream() vs invoke(): stream() yields one {node_name: output} dict per node
# so you can trace Command routing live; invoke() blocks until END.
for event in app.stream({"task": task, "route": "", "result": ""}):
    for node_name, node_output in event.items():
        print(f"  NODE: {node_name}")
        if isinstance(node_output, dict):
            for k, v in node_output.items():
                preview = str(v)[:80].replace("\n", " ")
                suffix = "..." if len(str(v)) > 80 else ""
                print(f"    {k}: {preview}{suffix}")
        else:
            print(f"    output: {node_output}")
    print()

In [ ]:
# ─── 7-B: Validate that all goto targets are registered nodes ────────────────

def validate_command_graph(b, valid_routes: set) -> None:
    """Verify every routing target is registered as a node."""
    registered = set(b.nodes.keys())
    print(f"Registered nodes : {sorted(registered)}")
    print(f"Expected routes  : {sorted(valid_routes)}")

    missing = valid_routes - registered
    if missing:
        print(f"\nERROR: goto targets not registered: {missing}")
        print("  Fix: call builder.add_node(name, fn) for each.")
    else:
        print("\nAll goto targets are registered. Graph is valid.")

    entry = next((target for source, target in b.edges if source == "__start__"), None)
    if entry in registered:
        print(f"Entry point {entry!r} is valid.")
    else:
        print(f"ERROR: Entry point {entry!r} is NOT a registered node.")


validate_command_graph(builder, VALID_ROUTES)

---

## Part 8 ★ — Advanced: Confidence Routing

---

### Routing by Confidence Score

Extend the router to output both a category and a confidence score. If confidence falls below a threshold, route to a **clarification** node that asks for more context before proceeding:

```
START
  │
  ▼
┌────────────────────┐
│  confident_router  │
└────────┬───────────┘
         │
         ├── confidence >= 0.7 ──► code | explain | math | creative
         │
         └── confidence < 0.7  ──► clarify ──► END
```

This pattern is useful in production when the classifier prompt may receive ambiguous input and silently routing to the wrong handler would produce a poor user experience.

In [ ]:
# ─── 8-A: Confidence-aware router ────────────────────────────────────────────
from typing import TypedDict

from langgraph.graph import END, StateGraph
from langgraph.types import Command

CONFIDENT_ROUTER_PROMPT = """Classify this task into exactly one category: code, explain, math, or creative.
Also rate your confidence from 0.0 to 1.0.

Task: {task}

Respond in exactly this format (two lines only):
category: <one of: code, explain, math, creative>
confidence: <number between 0.0 and 1.0>"""


class ConfidentState(TypedDict):
    task: str
    route: str
    confidence: float
    result: str


def confident_router(state: ConfidentState) -> Command:
    raw = llm.invoke([HumanMessage(content=CONFIDENT_ROUTER_PROMPT.format(task=state["task"]))]).content

    route = "explain"
    confidence = 0.5
    for line in raw.strip().splitlines():
        if line.startswith("category:"):
            route = safe_route(line.split(":", 1)[1].strip())
        elif line.startswith("confidence:"):
            try:
                confidence = float(line.split(":", 1)[1].strip())
            except ValueError:
                confidence = 0.5

    print(f"  [router] route={route!r}, confidence={confidence:.2f}")
# 0.7 threshold is a tunable dial: lower it to trust the LLM more often,
# raise it to escalate ambiguous tasks to human clarification more aggressively.
    next_node = "clarify" if confidence < 0.7 else route
    return Command(goto=next_node, update={"route": route, "confidence": confidence})


def clarify(state: ConfidentState) -> dict:
    """Low-confidence fallback — return a clarifying question."""
    result = llm.invoke([
        SystemMessage(content=(
            "The user's request is ambiguous. Ask one specific clarifying question "
            "to determine whether this is a coding, explanation, math, or creative task."
        )),
        HumanMessage(content=state["task"]),
    ]).content
    return {"result": f"[CLARIFICATION NEEDED] {result}"}


c_builder = StateGraph(ConfidentState)
c_builder.add_node("router", confident_router)
c_builder.add_node("clarify", clarify)
for cat in VALID_ROUTES:
    c_builder.add_node(cat, make_handler(cat))
    c_builder.add_edge(cat, END)
c_builder.add_edge("clarify", END)
c_builder.set_entry_point("router")
c_app = c_builder.compile()

for task in [
    "Write a bubble sort algorithm in Python.",  # clear → code
    "Tell me something interesting.",            # ambiguous → clarify
]:
    print(f"\nTASK: {task}")
    r = c_app.invoke({"task": task, "route": "", "confidence": 0.0, "result": ""})
    print(f"Route: {r['route']} (confidence={r['confidence']:.2f})")
    print(f"Result: {r['result'][:200]}")

---

## Exercises

---

### Exercise 1 — Add a Fifth Category

Extend the Part 3 graph with a `"summarise"` category that condenses any text into 3 bullet points.

**Steps:**
1. Add `"summarise"` to `VALID_ROUTES` and `PROMPTS`
2. Update `ROUTER_PROMPT` to include the new category
3. Add a `summarise_handler` node and wire it to `END`
4. Test with `"Summarise: The French Revolution began in 1789..."`

**Key insight:** With Command, adding a category touches only 2 things (routes set + node list). With `add_conditional_edges` it would be 3 (routes set + edge function + mapping dict).

---

### Exercise 2 — Multi-hop with Retry Validation

Build a graph where `code_handler` writes a draft, then routes to `validate`. The validator checks if the draft contains `def ` (a Python function definition). If yes, route to `END`. If no and `retries < 2`, route back to `code_handler` for a retry.

**Key insight:** A cycle in a Command graph is fine as long as state evolves on each pass and termination is guaranteed by a counter.

---

### Exercise 3 — Fan-out Summarisation with Send

Given `DOCUMENTS = ["doc1", "doc2", "doc3"]`, use `Send` to dispatch each document to a `summarise_doc` node in parallel, then collect all summaries into `summaries: Annotated[list, operator.add]`.

**Key insight:** Without `Annotated[list, operator.add]`, only one summary survives — the last branch write overwrites all others.

---

### Exercise 4 — Research: `Command(resume=value)` vs `Command(goto=...)`

Read the LangGraph docs on `interrupt()` and `Command(resume=value)`. In your own words:
- When would you use `Command(resume=value)` vs `Command(goto="next_node")`?
- Sketch (as a comment or markdown cell) a graph where the router pauses for human approval before routing.

In [ ]:
# ── Exercise 1 Starter — Add a "summarise" category ──────────────────────────

# TODO: extend VALID_ROUTES to include "summarise"
EX1_ROUTES = {"code", "explain", "math", "creative"}  # add "summarise" here

# TODO: add a summarise prompt
EX1_PROMPTS = {
    **PROMPTS,
    # "summarise": "You are a professional summariser. Condense the following into 3 bullet points: {task}",
}

# TODO: update ROUTER_PROMPT to list five categories
EX1_ROUTER_PROMPT = ROUTER_PROMPT  # replace with updated prompt

# TODO: rebuild the graph and test
print("Exercise 1: add your implementation here.")

In [ ]:
# ── Exercise 2 Starter — Multi-hop with retry validation ─────────────────────
from typing import TypedDict

from langgraph.graph import END, StateGraph
from langgraph.types import Command


class RetryState(TypedDict):
    task: str
    draft: str
    retries: int
    result: str


MAX_RETRIES = 2


def code_with_retry(state: RetryState) -> Command:
    # TODO: generate draft with llm
    # TODO: return Command(goto="validate", update={"draft": draft})
    pass


def validate(state: RetryState) -> Command:
    # TODO: check if draft contains "def "
    # TODO: if valid or max retries: Command(goto=END, update={"result": draft})
    # TODO: else: Command(goto="code", update={"retries": state["retries"] + 1})
    pass


# TODO: build graph and invoke
print("Exercise 2: add your implementation here.")

In [ ]:
# ── Exercise 3 Starter — Fan-out summarisation with Send ──────────────────────
import operator
from typing import Annotated, TypedDict

from langgraph.graph import END, StateGraph
from langgraph.types import Command, Send

DOCUMENTS = [
    "The Python programming language was created by Guido van Rossum in 1991.",
    "LangGraph is a framework for building stateful multi-agent applications.",
    "Command lets nodes control their own routing via goto and update.",
]


class FanOutState(TypedDict):
    documents: list
    doc: str
    summaries: Annotated[list, operator.add]  # reducer: accumulate branch outputs


def fan_out_dispatcher(state: FanOutState) -> Command:
    # TODO: return Command(goto=[Send("summarise_doc", {"doc": d, ...}) for d in state["documents"]])
    pass


def summarise_doc(state: FanOutState) -> dict:
    # TODO: summarise state["doc"] using llm
    # TODO: return {"summaries": [one_sentence_summary]}
    pass


# TODO: build graph, invoke with {"documents": DOCUMENTS, "doc": "", "summaries": []}
print("Exercise 3: add your implementation here.")

---

## Answer Key

Use this section as a reference after attempting the exercises yourself. These are sample solutions, not the only correct answers.

---

### Exercise 1 — Add a Fifth Category

**What to change:** Add `"summarise"` to `VALID_ROUTES` and `PROMPTS`, update the router prompt to list five categories, add a node and edge to `END`.

**Key insight:** Adding a category in the Command pattern requires touching only two things — the routes set and the node list. With `add_conditional_edges` there would be a third: the mapping dict. The Command pattern scales better as the number of categories grows.

In [ ]:
# Answer Key — Exercise 1
from langchain_core.messages import HumanMessage, SystemMessage
from langgraph.graph import END, StateGraph
from langgraph.types import Command

EX1_ROUTES = {"code", "explain", "math", "creative", "summarise"}

EX1_PROMPTS = {
    **PROMPTS,
    "summarise": "You are a professional summariser. Condense the following into exactly 3 concise bullet points: {task}",
}

EX1_ROUTER_PROMPT = (
    "Classify this task into exactly one category: "
    "code, explain, math, creative, or summarise.\n\n"
    "Task: {task}\n\n"
    "Respond with only the category name (no punctuation, no explanation)."
)


def ex1_router(state: CommandState) -> Command:
    raw = llm.invoke([HumanMessage(content=EX1_ROUTER_PROMPT.format(task=state["task"]))]).content
    route = raw.strip().lower()
    if route not in EX1_ROUTES:
        route = "explain"
    print(f"  [router] → {route!r}")
    return Command(goto=route, update={"route": route})


def ex1_make_handler(cat):
    def h(state: CommandState) -> dict:
        return {
            "result": llm.invoke(
                [SystemMessage(content=EX1_PROMPTS[cat].format(task=state["task"]))]
            ).content
        }
    h.__name__ = f"{cat}_handler"
    return h


ex1_builder = StateGraph(CommandState)
ex1_builder.add_node("router", ex1_router)
for cat in EX1_ROUTES:
    ex1_builder.add_node(cat, ex1_make_handler(cat))
    ex1_builder.add_edge(cat, END)
ex1_builder.set_entry_point("router")
ex1_app = ex1_builder.compile()

task = "Summarise: The French Revolution began in 1789, led to the execution of King Louis XVI, and ended with Napoleon's rise to power."
r = ex1_app.invoke({"task": task, "route": "", "result": ""})
print(f"Route : {r['route']}")
print(f"Result:\n{r['result']}")

### Exercise 2 — Multi-hop with Retry Validation

**Key pattern:** A cycle in a Command graph is safe when the state evolves on each pass and a counter guarantees termination. The validate node inspects the draft and routes back to `code` only if the draft is invalid AND `retries < MAX_RETRIES`. The counter increments on each retry — without it, you would have an unbounded loop.

In [ ]:
# Answer Key — Exercise 2
from typing import TypedDict

from langchain_core.messages import SystemMessage
from langgraph.graph import END, StateGraph
from langgraph.types import Command


class RetryState(TypedDict):
    task: str
    draft: str
    retries: int
    result: str


MAX_RETRIES = 2


def code_with_retry(state: RetryState) -> Command:
    print(f"  [code_handler] attempt {state['retries'] + 1}")
    draft = llm.invoke(
        [SystemMessage(content=f"Write Python code for: {state['task']}")]
    ).content
    return Command(goto="validate", update={"draft": draft})


def validate(state: RetryState) -> Command:
    has_function = "def " in state["draft"]
    print(f"  [validate] has function def: {has_function}, retries: {state['retries']}")

    if has_function or state["retries"] >= MAX_RETRIES:
        label = "valid" if has_function else "max retries reached"
        print(f"  [validate] done ({label})")
        return Command(goto=END, update={"result": state["draft"]})
    else:
        return Command(goto="code", update={"retries": state["retries"] + 1})


r2_builder = StateGraph(RetryState)
r2_builder.add_node("code", code_with_retry)
r2_builder.add_node("validate", validate)
r2_builder.set_entry_point("code")
r2_app = r2_builder.compile()

out = r2_app.invoke({"task": "fibonacci function", "draft": "", "retries": 0, "result": ""})
print(f"\nFinal result (first 300 chars):\n{out['result'][:300]}")

### Exercise 3 — Fan-out Summarisation with Send

**Key pattern:** `Annotated[list, operator.add]` is the reducer that accumulates results from parallel branches. Without it, each branch overwrites the same field and only the last write survives. LangGraph applies `operator.add` automatically when multiple nodes write to an `Annotated` field.

In [ ]:
# Answer Key — Exercise 3
import operator
from typing import Annotated, TypedDict

from langchain_core.messages import HumanMessage, SystemMessage
from langgraph.graph import END, StateGraph
from langgraph.types import Command, Send

DOCUMENTS = [
    "The Python programming language was created by Guido van Rossum in 1991.",
    "LangGraph is a framework for building stateful multi-agent applications.",
    "Command lets nodes control their own routing via goto and update.",
]


class FanOutState(TypedDict):
    documents: list
    doc: str
    summaries: Annotated[list, operator.add]


def fan_out_dispatcher(state: FanOutState) -> Command:
    return Command(
        goto=[
            Send("summarise_doc", {"doc": doc, "documents": [], "summaries": []})
            for doc in state["documents"]
        ]
    )


def summarise_doc(state: FanOutState) -> dict:
    summary = llm.invoke([
        SystemMessage(content="Summarise in exactly one sentence."),
        HumanMessage(content=state["doc"]),
    ]).content.strip()
    print(f"  [summarise_doc] '{state['doc'][:40]}...' → '{summary[:60]}'")
    return {"summaries": [summary]}


fo_builder = StateGraph(FanOutState)
fo_builder.add_node("dispatcher", fan_out_dispatcher)
fo_builder.add_node("summarise_doc", summarise_doc)
fo_builder.add_edge("summarise_doc", END)
fo_builder.set_entry_point("dispatcher")
fo_app = fo_builder.compile()

result = fo_app.invoke({"documents": DOCUMENTS, "doc": "", "summaries": []})
print(f"\nCollected {len(result['summaries'])} summaries:")
for i, s in enumerate(result["summaries"], 1):
    print(f"  {i}. {s}")

### Exercise 4 — `Command(resume=value)` vs `Command(goto=...)`

| | `Command(goto=...)` | `Command(resume=value)` |
|---|---|---|
| **When** | Node finishes and routes to a new node | Graph was paused by `interrupt()` and human provides input |
| **Who calls it** | The *node function* returns it | The *caller* (your app) passes it to `app.invoke()` or `app.stream()` |
| **Effect** | Writes `update` to state, moves to `goto` | Resumes the interrupted node; `interrupt()` returns the `resume` value |

**Sketch of a HITL approval graph:**

```python
from langgraph.types import interrupt, Command

def router_with_approval(state) -> Command:
    route = classify(state["task"])

    # Pause here — LangGraph checkpoints the graph and waits
    approved = interrupt({"proposed_route": route, "task": state["task"]})

    # `approved` is whatever the caller passed as Command(resume=...)
    final_route = route if approved else "explain"   # human can override
    return Command(goto=final_route, update={"route": final_route})


# External caller resumes after human responds:
# thread_config = {"configurable": {"thread_id": "1"}}
# app.invoke(Command(resume=True), config=thread_config)
```

See example `11-hitl-approval` in this repo for a complete working implementation with `SqliteSaver` checkpointing.

---

## What's Next?

You now understand the Command pattern end-to-end — from basic string routing to fan-out with `Send`, multi-hop chains, and confidence-based dispatch.

### Human-in-the-Loop
- **Example 11 — HITL Approval** (`../11-hitl-approval/`): Combine `interrupt()` with `Command(resume=value)` to pause the router built here for human review before routing. The Command pattern makes the approval gate trivial to add.

### Persistence Across Sessions
- **Example 39 — Checkpoint Resume** (`../39-checkpoint-resume/`): Use `SqliteSaver` to persist graph state across Python sessions. Combined with Command routing, this gives you durable multi-step pipelines that survive process restarts.

### Multi-Agent Systems
- **Example 6 — Multi-Agent Graph** (`../6-multi-agent-graph/`): Apply `Command(goto=...)` across subgraph boundaries. The supervisor pattern — one controller node routing between specialist agents — is a direct extension of the router built in Part 3.

### Fan-out at Scale
- **Example 30 — Agentic RAG** (`../30-agentic-rag/`): The `Send`-based fan-out from Part 4 is the same mechanism used in map-reduce RAG pipelines, where each document is processed in parallel and results are collected through reducers.

### Further Reading
- LangGraph `Command` reference: https://langchain-ai.github.io/langgraph/reference/types/#langgraph.types.Command
- LangGraph `Send` reference: https://langchain-ai.github.io/langgraph/reference/types/#langgraph.types.Send
- LangGraph how-to — dynamic routing with Command: https://langchain-ai.github.io/langgraph/how-tos/command/
- LangGraph how-to — map-reduce with Send: https://langchain-ai.github.io/langgraph/how-tos/map-reduce/
- LangGraph conceptual guide — multi-agent handoffs: https://langchain-ai.github.io/langgraph/concepts/multi_agent/

---

*Workshop complete. The natural next step is example 11 — add a human approval gate to the router you just built.*